# 🏥 KG-CMI: Knowledge Graph Enhanced Cross-Mamba Interaction for Med-VQA
### 🎓 Minor Project Training Notebook (Google Colab / Cloud GPU)

## Step 1: Hardware Check & Repository Cloning

In [ ]:
!nvidia-smi
import os, shutil
os.chdir("/content")
if os.path.exists("/content/Med-VQA"):
    shutil.rmtree("/content/Med-VQA")
!git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
%cd /content/Med-VQA
print("✅ Repository cloned to /content/Med-VQA!")

## Step 2: Install All Required Dependencies (exact pinned versions)

In [ ]:
# Install required dependencies with --prefer-binary to prevent build hangs
!pip install --prefer-binary -q timm==0.4.12 pytorch-lightning==1.9.5 transformers==4.35.2 torchmetrics==0.7.2
!pip install --prefer-binary -q einops sacred pandas ftfy scikit-learn pyarrow datasets
!pip install --prefer-binary -q fairscale iopath omegaconf webdataset opendatasets sentencepiece decord pycocotools
print("✅ All dependencies installed successfully!")

## Step 2.5: Apply Compatibility Patches

In [ ]:
import os, re

bert_paths = [
    "/content/Med-VQA/m3ae/modules/language_encoders/bert_model.py",
    "m3ae/modules/language_encoders/bert_model.py"
]
patched = False
for bp in bert_paths:
    if os.path.exists(bp):
        with open(bp, "r") as f:
            content = f.read()
        if "tokenizer_class=" in content:
            content = re.sub(r"\s*tokenizer_class=_TOKENIZER_FOR_DOC,", "", content)
            with open(bp, "w") as f:
                f.write(content)
            print(f"✅ {bp} patched!")
        else:
            print(f"✅ {bp} is clean!")
        patched = True
        break
if not patched:
    print("⚠️ Repository directory not found. Please run Step 1 first.")

## Step 3: Mount Google Drive

In [ ]:
import sys, os
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA

os.chdir("/content/Med-VQA")
if "/content/Med-VQA" not in sys.path:
    sys.path.insert(0, "/content/Med-VQA")

import torch
import pytorch_lightning as pl
import timm
import fairscale
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ PyTorch Lightning: {pl.__version__}")
print(f"✅ timm: {timm.__version__}")
print(f"✅ fairscale: {fairscale.__version__}")

import torch.nn as nn

class RelationalGraphConvLayer(nn.Module):
    def __init__(self, in_features, out_features, num_relations=4, num_bases=4):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.num_relations = num_relations
        self.num_bases = num_bases
        self.weight_self = nn.Parameter(torch.Tensor(in_features, out_features))
        self.bases = nn.Parameter(torch.Tensor(num_bases, in_features, out_features))
        self.rel_coeffs = nn.Parameter(torch.Tensor(num_relations, num_bases))
        self.bias = nn.Parameter(torch.Tensor(out_features))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight_self)
        nn.init.xavier_uniform_(self.bases)
        nn.init.xavier_uniform_(self.rel_coeffs)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, edge_type):
        N_v = x.size(0)
        out = torch.matmul(x, self.weight_self)
        W_r = torch.einsum("rb, bio -> rio", self.rel_coeffs, self.bases)
        src, dst = edge_index[0], edge_index[1]
        for r in range(self.num_relations):
            rel_mask = (edge_type == r)
            if not rel_mask.any():
                continue
            r_src, r_dst = src[rel_mask], dst[rel_mask]
            msg = torch.matmul(x[r_src], W_r[r])
            deg = torch.zeros(N_v, device=x.device).scatter_add_(0, r_dst, torch.ones_like(r_dst, dtype=torch.float)).clamp(min=1.0)
            norm = 1.0 / deg[r_dst].unsqueeze(-1)
            out.scatter_add_(0, r_dst.unsqueeze(-1).expand(-1, self.out_features), msg * norm)
        return out + self.bias

print("✅ Upgraded R-GCN module ready!")

# Download & preprocess full VQA-RAD dataset automatically
os.chdir("/content/Med-VQA")
!python download_and_prep_vqa_rad.py

## Step 4: Verify Environment & Download Full VQA-RAD Dataset

In [ ]:
import os
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
os.chdir("/content/Med-VQA")

save_dir = CHECKPOINT_DIR if "CHECKPOINT_DIR" in locals() else "/content/checkpoints"
os.makedirs(save_dir, exist_ok=True)
print(f"Saving logs and checkpoints to: {save_dir}")

!python main.py with \n    data_root=data/finetune_arrows/ \n    num_gpus=1 num_nodes=1 gpu_ids=0 \n    task_finetune_vqa_vqa_rad \n    per_gpu_batchsize=8 \n    image_size=384 \n    max_epoch=20 \n    log_dir=

## Step 5: Train & Evaluate Upgraded Model on Full VQA-RAD

In [ ]:
import os
if not os.path.exists("/content/Med-VQA"):
    os.chdir("/content")
    !git clone https://github.com/Anjali-2807/Med-VQA.git /content/Med-VQA
os.chdir("/content/Med-VQA")

save_dir = CHECKPOINT_DIR if "CHECKPOINT_DIR" in locals() else "/content/checkpoints"

!python main.py with \n    data_root=data/finetune_arrows/ \n    num_gpus=1 num_nodes=1 gpu_ids=0 \n    task_finetune_vqa_vqa_rad \n    per_gpu_batchsize=8 \n    image_size=384 \n    test_only=True \n    log_dir=

## Step 6: Test Only Mode (Evaluate Checkpoint)

In [ ]:
%cd /content/Med-VQA
import os
save_dir = CHECKPOINT_DIR if "CHECKPOINT_DIR" in locals() else "/content/checkpoints"

!python main.py with \n    data_root=data/finetune_arrows/ \n    num_gpus=1 num_nodes=1 gpu_ids=0 \n    task_finetune_vqa_vqa_rad \n    per_gpu_batchsize=8 \n    image_size=384 \n    test_only=True \n    log_dir=